# OE2B-06 — Evaluación k-hive-fold

## ¿Qué hace este notebook?

Evalúa formalmente los modelos xnorC entrenados en OE2B-05 usando los resultados
del k-hive-fold. Para cada experimento (B1–B4) calcula:

- **Accuracy, F1, AUC** por fold y promedio ± std
- **Matriz de confusión** acumulada (suma de todos los folds)
- **Curva ROC** por experimento
- **Tabla comparativa** B1 vs B2 vs B3 vs B4

Estos resultados van directo a la tesis como evidencia del desempeño de xnorC
en colmenas que el modelo nunca vio durante el entrenamiento.

### Inputs
- `../data/processed/modelos/xnorC_B{n}_fold{k}.pt` — modelos de OE2B-05
- `../data/processed/pares/folds/fold_{k}_test.csv` — pares de test de OE2B-03
- `../data/processed/features/features_B{n}_*.npy` — features de OE2B-04

### Outputs
- `../data/processed/resultados/evaluacion_khivefold.csv` — tabla completa
- `../data/processed/resultados/grafico_roc.png`
- `../data/processed/resultados/grafico_confusion.png`

## 1. Importaciones y configuración

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve
)

print("Importaciones listas")

In [ ]:
CARPETA_FOLDS      = Path('../data/processed/pares/folds')
CARPETA_FEATURES   = Path('../data/processed/features')
CARPETA_MODELOS    = Path('../data/processed/modelos')
CARPETA_RESULTADOS = Path('../data/processed/resultados')
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cpu')
BATCH_SIZE = 64

EXPERIMENTOS = {
    'B1': 'features_B1_mel.npy',
    'B2': 'features_B2_mfcc.npy',
    'B3': 'features_B3_fft.npy',
    'B4': 'features_B4_cqt.npy',
}

print("Configuración lista")

## 2. Cargar arquitectura y features

Copiamos la definición del modelo de OE2B-05 para poder cargar los `.pt` guardados.

In [ ]:
# ── Definición de la red (igual que OE2B-05) ──────────────────────────────────

class Encoder(nn.Module):
    def __init__(self, dim_entrada, dim_embedding=64, dropout=0.3):
        super().__init__()
        dim_oculta = dim_embedding * 2
        self.red = nn.Sequential(
            nn.Linear(dim_entrada, dim_oculta), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(dim_oculta, dim_embedding), nn.ReLU(),
        )
    def forward(self, x):
        return self.red(x)

class Clasificador(nn.Module):
    def __init__(self, dim_embedding=64, dropout=0.3):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(dim_embedding, 32), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 2)
        )
    def forward(self, diff):
        return self.red(diff)

class xnorC(nn.Module):
    def __init__(self, dim_entrada, dim_embedding=64, dropout=0.3):
        super().__init__()
        self.encoder      = Encoder(dim_entrada, dim_embedding, dropout)
        self.clasificador = Clasificador(dim_embedding, dropout)
    def forward(self, a, b):
        return self.clasificador(torch.abs(self.encoder(a) - self.encoder(b)))

# ── Cargar features e índice ──────────────────────────────────────────────────

df_indice = pd.read_csv(CARPETA_FEATURES / 'features_index.csv')
archivo_a_indice = {
    Path(row['ruta']).name: idx
    for idx, row in df_indice.iterrows()
}

features_dict = {}
for nombre, archivo in EXPERIMENTOS.items():
    ruta = CARPETA_FEATURES / archivo
    if ruta.exists():
        features_dict[nombre] = np.load(ruta)
        print(f"  {nombre}: {features_dict[nombre].shape}")
    else:
        print(f"  ⚠️  No encontrado: {ruta}")

print(f"\nÍndice: {len(archivo_a_indice)} archivos")

## 3. Evaluar todos los modelos

Para cada experimento y cada fold:
1. Cargamos el modelo `.pt` de OE2B-05
2. Cargamos los pares de test del fold correspondiente
3. Calculamos predicciones y probabilidades
4. Guardamos métricas y predicciones crudas (para ROC y confusión acumulada)

In [ ]:
class DatasetPares(Dataset):
    """Igual que OE2B-05 — carga pares desde CSV con columnas ruta_a/ruta_b/etiqueta."""
    def __init__(self, df_pares, features_array, mapa):
        self.pares    = df_pares.reset_index(drop=True)
        self.features = features_array
        self.mapa     = mapa
        mascara = self.pares.apply(
            lambda f: Path(f['ruta_a']).name in mapa and Path(f['ruta_b']).name in mapa,
            axis=1
        )
        self.pares = self.pares[mascara].reset_index(drop=True)
    def __len__(self):
        return len(self.pares)
    def __getitem__(self, idx):
        f = self.pares.iloc[idx]
        return (
            torch.tensor(self.features[self.mapa[Path(f['ruta_a']).name]], dtype=torch.float32),
            torch.tensor(self.features[self.mapa[Path(f['ruta_b']).name]], dtype=torch.float32),
            torch.tensor(int(f['etiqueta']), dtype=torch.long)
        )

In [ ]:
def evaluar_modelo(modelo, loader, device):
    """Devuelve etiquetas reales, predicciones y probabilidades para clase 1 (IGUAL)."""
    modelo.eval()
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for a, b, etiq in loader:
            logits = modelo(a.to(device), b.to(device))
            probs  = torch.softmax(logits, dim=1)
            preds  = torch.argmax(logits, dim=1)
            y_true.extend(etiq.numpy())
            y_pred.extend(preds.cpu().numpy())
            y_prob.extend(probs[:, 1].cpu().numpy())
    return np.array(y_true), np.array(y_pred), np.array(y_prob)


# Detectar cuántos folds hay
n_folds = len(list(CARPETA_FOLDS.glob('fold_*_test.csv')))
print(f"Folds encontrados: {n_folds}")

# Ejecutar evaluación
todos_resultados  = []   # una fila por (experimento, fold)
predicciones_raw  = {}   # experimento → {y_true, y_pred, y_prob acumulados}

for exp, archivo_feat in EXPERIMENTOS.items():
    if exp not in features_dict:
        print(f"⚠️  Saltando {exp}: sin features")
        continue

    arr = features_dict[exp]
    predicciones_raw[exp] = {'y_true': [], 'y_pred': [], 'y_prob': []}

    print(f"\n{'─'*50}")
    print(f" {exp}")

    for k in range(n_folds):
        ruta_modelo = CARPETA_MODELOS / f'xnorC_{exp}_fold{k}.pt'
        ruta_test   = CARPETA_FOLDS   / f'fold_{k}_test.csv'

        if not ruta_modelo.exists():
            print(f"  Fold {k}: ⚠️  modelo no encontrado")
            continue

        # Cargar modelo
        ckpt   = torch.load(ruta_modelo, map_location='cpu')
        modelo = xnorC(ckpt['dim_entrada'], ckpt['embedding_dim'], ckpt['dropout'])
        modelo.load_state_dict(ckpt['estado_modelo'])
        modelo.to(DEVICE)

        # Cargar pares de test
        df_test = pd.read_csv(ruta_test)
        ds_test = DatasetPares(df_test, arr, archivo_a_indice)
        loader  = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False)

        # Evaluar
        y_true, y_pred, y_prob = evaluar_modelo(modelo, loader, DEVICE)

        acc = accuracy_score(y_true, y_pred)
        f1  = f1_score(y_true, y_pred, average='weighted')
        auc = roc_auc_score(y_true, y_prob)

        print(f"  Fold {k}: acc={acc:.3f}  f1={f1:.3f}  auc={auc:.3f}  (n={len(y_true)})")

        todos_resultados.append({
            'experimento': exp,
            'fold': k,
            'n_pares': len(y_true),
            'accuracy': round(acc, 4),
            'f1': round(f1, 4),
            'auc': round(auc, 4),
        })

        # Acumular predicciones para gráficos globales
        predicciones_raw[exp]['y_true'].extend(y_true.tolist())
        predicciones_raw[exp]['y_pred'].extend(y_pred.tolist())
        predicciones_raw[exp]['y_prob'].extend(y_prob.tolist())

print("\nEvaluación completa.")

## 4. Tabla resumen

In [ ]:
df_res = pd.DataFrame(todos_resultados)

print("Resultados por fold:")
print(df_res.to_string(index=False))
print()

# Promedio ± std por experimento
resumen = (
    df_res.groupby('experimento')[['accuracy', 'f1', 'auc']]
    .agg(['mean', 'std'])
    .round(3)
)
print("Promedio ± std por experimento:")
print(resumen)

# Guardar CSV
df_res.to_csv(CARPETA_RESULTADOS / 'evaluacion_khivefold.csv', index=False)
print(f"\nGuardado: {CARPETA_RESULTADOS / 'evaluacion_khivefold.csv'}")

## 5. Matrices de confusión acumuladas

Sumamos las matrices de confusión de todos los folds para cada experimento.
Esto da una imagen global del desempeño sin depender de un solo fold.

In [ ]:
exps_disponibles = [e for e in EXPERIMENTOS if e in predicciones_raw and predicciones_raw[e]['y_true']]
n_exp = len(exps_disponibles)

if n_exp == 0:
    print("No hay predicciones para graficar.")
else:
    fig, axes = plt.subplots(1, n_exp, figsize=(4 * n_exp, 4))
    if n_exp == 1:
        axes = [axes]

    etiquetas_clases = ['DIFERENTE\n(0)', 'IGUAL\n(1)']

    for ax, exp in zip(axes, exps_disponibles):
        raw   = predicciones_raw[exp]
        cm    = confusion_matrix(raw['y_true'], raw['y_pred'])
        # Normalizar por fila (porcentaje sobre el total de cada clase real)
        cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

        im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(etiquetas_clases, fontsize=9)
        ax.set_yticklabels(etiquetas_clases, fontsize=9)
        ax.set_xlabel('Predicho', fontsize=9)
        ax.set_ylabel('Real', fontsize=9)
        ax.set_title(f'{exp}', fontsize=11)

        # Mostrar valor en cada celda
        for i in range(2):
            for j in range(2):
                color = 'white' if cm_pct[i, j] > 60 else 'black'
                ax.text(j, i, f'{cm_pct[i,j]:.1f}%\n({cm[i,j]})',
                        ha='center', va='center', fontsize=10, color=color)

    fig.suptitle('Matrices de confusión — k-hive-fold acumulado (todos los folds)', fontsize=12)
    plt.tight_layout()
    plt.savefig(CARPETA_RESULTADOS / 'grafico_confusion.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Guardado: {CARPETA_RESULTADOS / 'grafico_confusion.png'}")

## 6. Curvas ROC

In [ ]:
if exps_disponibles:
    fig, ax = plt.subplots(figsize=(7, 6))

    colores = {'B1': '#1565C0', 'B2': '#6A1B9A', 'B3': '#E65100', 'B4': '#2E7D32'}

    for exp in exps_disponibles:
        raw = predicciones_raw[exp]
        fpr, tpr, _ = roc_curve(raw['y_true'], raw['y_prob'])
        auc_val = roc_auc_score(raw['y_true'], raw['y_prob'])
        ax.plot(fpr, tpr,
                label=f'{exp} (AUC={auc_val:.3f})',
                color=colores.get(exp, 'gray'), linewidth=2)

    # Línea de azar
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Azar (AUC=0.5)')

    ax.set_xlabel('Tasa de falsos positivos', fontsize=11)
    ax.set_ylabel('Tasa de verdaderos positivos', fontsize=11)
    ax.set_title('Curvas ROC — k-hive-fold (todos los folds acumulados)', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig(CARPETA_RESULTADOS / 'grafico_roc.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Guardado: {CARPETA_RESULTADOS / 'grafico_roc.png'}")

## 7. Comparación por fold (boxplot)

In [ ]:
if not df_res.empty:
    fig, axes = plt.subplots(1, 3, figsize=(13, 5))

    for ax, metrica in zip(axes, ['accuracy', 'f1', 'auc']):
        datos = [df_res[df_res['experimento'] == exp][metrica].values
                 for exp in ['B1', 'B2', 'B3', 'B4']]
        bp = ax.boxplot(datos, labels=['B1\nMel', 'B2\nMFCC', 'B3\nFFT', 'B4\nCQT'],
                        patch_artist=True)
        colores_box = ['#BBDEFB', '#E1BEE7', '#FFE0B2', '#C8E6C9']
        for patch, color in zip(bp['boxes'], colores_box):
            patch.set_facecolor(color)
        ax.axhline(0.5, color='red', linestyle='--', alpha=0.4, label='azar')
        ax.set_title(metrica.upper(), fontsize=11)
        ax.set_ylim(0, 1)
        ax.grid(axis='y', alpha=0.3)
        ax.legend(fontsize=8)

    fig.suptitle('xnorC — Distribución de métricas por fold (k-hive-fold)', fontsize=12)
    plt.tight_layout()
    plt.savefig(CARPETA_RESULTADOS / 'grafico_boxplot_khivefold.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Guardado: {CARPETA_RESULTADOS / 'grafico_boxplot_khivefold.png'}")